In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import logging

logging.basicConfig(level=logging.INFO)

In [4]:
import os
from openai import AsyncOpenAI
from aiolimiter import AsyncLimiter
from asyncio import Semaphore
from math_verify import parse
from dataclasses import dataclass

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
MODEL_ID = "openai/gpt-oss-20b"

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

@dataclass
class MyCompletionChoice:
	reasoning_content: str = ""
	content: str = ""

limiter = AsyncLimiter(16)
semaphore = Semaphore(128)
async def create_completion(*args, **kwargs):
	kwargs["stream"] = True
	while True:
		try:
			async with limiter:
				async with semaphore:
					choices: list[MyCompletionChoice] = []
					async for chunk in await client.chat.completions.create(*args, **kwargs):
						for choice in chunk.choices:
							while len(choices) <= choice.index:
								choices.append(MyCompletionChoice())

							if delta := getattr(choice.delta, "reasoning_content", None):
								choices[choice.index].reasoning_content += delta

							if delta := getattr(choice.delta, "content", None):
								choices[choice.index].content += delta
					if choices:
						return choices
		except:
			pass

prompt = "What is 13 times 17? Box your answer."
gold = "221"

choices = await create_completion(
	model=MODEL_ID,
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(choices):
  parsed_answer = parse(choice.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Choice {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(f"<think>{choice.reasoning_content}</think>{choice.content}")

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Choice 1: 221 (correct) ********************
<think>We need to answer 13 times 17, and box the answer. They want 13 * 17 = 221. Box answer meaning probably put in a box like by using brackets or something. Can use markdown to show box, maybe like

```
┌───────┐
│ 221   │
└───────┘
```

or use LaTeX \boxed{221}. Provide answer inside box. Probably best to use \boxed{221}. Let's answer.

</think>\[
\boxed{221}
\]
******************** Choice 2: 221 (correct) ********************
<think>The user asks: "What is 13 times 17? Box your answer." Likely expecting the result: 221. And they want it boxed. Maybe like a box around the answer. We should show the answer in a box. We can use LaTeX or plain text. For BB markup: Use something like:

```
+-----+
| 221 |
+-----+
```

Alternatively use a TeX box: \boxed{221}. We need to interpret the instruction: "Box your answer." They want a simple 

In [5]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")
ds

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/open-r1/OpenR1-Math-220k/e4e141ec9dea9f8326f4d347be56105859b2bd68/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/open-r1/OpenR1-Math-220k/open-r1/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/revision/e4e141ec9dea9f8326f4d347be56105859b2bd68 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/.huggingface.yaml "HTTP/1.1 404 

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/tree/e4e141ec9dea9f8326f4d347be56105859b2bd68/data?recursive=true&expand=false "HTTP/1.1 200 OK"


Dataset({
    features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
    num_rows: 93733
})

In [6]:
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"results": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(create_completion(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	with logging_redirect_tqdm():
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, choices, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		results = []
		for choice in choices:
			answer = parse(choice.content)
			result = verify(gold, answer)

			outputs.append(f"<think>{choice.reasoning_content}</think>{choice.content}")
			results.append(result)

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["results"].append(results)
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model=MODEL_ID,
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [00:21<00:00, 10.93s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>The user: "Pick an random integer from 1 to 3. Don\'t pick 2. Box your answer." They want us to pick a random integer from 1 to 3 but not pick 2. So likely answer should be 1 or 3. "Box your answer." They likely want the answer inside a box. Maybe just put brackets? Or use markdown to create a box? Could use triple backticks but that is code block; maybe use ☐ or use a table. Perhaps use a box drawn with \'|\' and \'-\' ascii. Something like:\n\n```\n┌─┐\n│3│\n└─┘\n```\n\nWe can\'t pick 2. Random, so 1 or 3, say 3. So answer in a box and indicate random. Use the ASCII box around 3. Let\'s do that.</think>Here is the selected number in a box:\n\n```\n┌─┐\n│3│\n└─┘\n```\n',
   '<think>The user: "Pick a random integer from 1 to 3. Don\'t pick 2. Box your answer." They specifically ask to pick random integer from 1 to 3 but not 2. So we must outp

In [7]:
input_ds = ds.shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model=MODEL_ID,
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completion

Dataset({
    features: ['prompt', 'outputs', 'results'],
    num_rows: 128
})

In [8]:
output_ds.push_to_hub("EthanKim8683/reg_grpo", "128")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo "HTTP/1.1 200 OK"
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

INFO:httpx:HTTP Request: POST https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/revision/31412a5213ac3c14e3f0909b64b9d2e4c398a27f "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/tree/31412a5213ac3c14e3f0909b64b9d2e4c398a27f/128?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/tree/31412a5213ac3c14e3f0909b64b9d2e4c398a27f?recursive=false&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/datasets/EthanKim8683/reg_grpo/resolve/31412a5213ac3c14e3f0909b64b9d2e4c398a27f/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/EthanKim8683/reg_grpo/31412a5213ac3c14e3f0909b64b9d2e4c398a27f/README.md?%2Fdatasets%2FEthanKim8683%2Freg_grpo%2Fresolve%2

CommitInfo(commit_url='https://huggingface.co/datasets/EthanKim8683/reg_grpo/commit/81fd4bb2db8b9d3fcd98f3c89c4a4925027cdec5', commit_message='Upload dataset', commit_description='', oid='81fd4bb2db8b9d3fcd98f3c89c4a4925027cdec5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/EthanKim8683/reg_grpo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EthanKim8683/reg_grpo'), pr_revision=None, pr_num=None)